# Instrumental Variables

## Assumptions

1. **Relevance**: $Z$ must be correlated with the endogenous regressor $X$.
2. **Exclusion**: $Z$ must be uncorrelated with the error term $\varepsilon$

## Environment Configuration

Import dependencies

In [ ]:
from linearmodels.iv import IV2SLS
import pandas as pd
import statsmodels.api as sm

## Data

### About the experiment

**Objective**: Estimate the effect of **education** on **wages**.

If we regress Log Wages on Years of Schooling, the estimate is biased because ability
or family background affect both education and wages. We therefore need a source of
variation in education that is unrelated to ability or family background.

- Schools require children to have turned 6 by a December 31st to enter first grade.
- Students are required to stay in school until their 16th birthday.

Becuase of these two rules, a student born in Q1 (early in the year) cannot enroll in
first grade at the same time as someone born in Q4 (late in the year), and when both
students reach their 16th birthday, the student born in Q4 will have completed more
years of schooling than the Q1 student.

**The Instrument**: Quarter of Birth (QOB)

First Stage (Relevance): QOB predicts total years of schooling. Students born earlier
in the year consistently have slightly less education on average because they hit the
legal dropout age earlier in their academic career.

Reduced Form: If education truly increases wages, we should see that people born in Q4
earn more than those born in Q1, simply because the law forced them to stay in school
longer.

The Exclusion Restriction: For this to work, we must assume that being born in December
vs. January has no direct effect on your earnings 20 years later, other than through
the extra time spent in school. This independence is essentially impossible to
demonstrate, but it seems reasonable since your month of birth is unlikely to have a
direct causal effect on your adult salary.

### Dictionary

- **log_weekly_wage**: Natural logarithm of the log weekly wage
- **education**: Years of education
- **qob**: Quarter of birth
- **age**: Age
- **is_married**: Indicates if the person is married
- **is_black**: Indicates if the person if Afroamerican
- **is_metropolitan**: Indicates if the person lives in a metropolitan area
- **region**: Indicates the region a person lives in
- **yob**: Year of birth

In [ ]:
# Load data
URL = "https://raw.githubusercontent.com/ArturoSbr/appec-s2-2025/refs/heads/main/lectures/data/angrist_krueger.csv"
df = pd.read_csv(URL)
display(df.head())

### Setting up the data

We will estimate a fixed effects model to account for unobserved time-invariant
confounders. To do this, we will use dummies, though as we will see in the next
lecture, we could use fixed effects.

In [ ]:
# Convert columns to dummies
df = pd.get_dummies(df, columns=['qob', 'region'], dtype='int')

# Add constant term
df['const'] = 1

# View result
display(df.head())

# Store names of fixed effects columns so we don't have to type them manually
fixed_effects = [
    col for col in df.columns if 'region_' in col
    and col != 'region_enocent'  # Omit one region
]

# Store names of controls too
controls = ['const', 'age', 'is_married', 'is_black', 'is_metropolitan']

## Naive Regression

We know that education is endogenous because one's IQ, ability, social skills, family
status and other unobservable factors have an effect on **both** wages **and**
education level. If we could observe these unobservable factors (and assuming we could
measure them without error), a naive regression model would correctly measure the
causal effect of education on wages. That's not the world we live in though!

In [ ]:
# Define spec with endogeneity (bad)
m0 = sm.OLS(
    endog=df['log_weekly_wage'],
    exog=df[['education'] + controls + fixed_effects]    
).fit(cog_type='HC1')

# Store results
education_naive = m0.params['education']
print(f'Naive estimated effect: {education_naive:.3f}')

## Homemade IV2SLS

The term 2SLS stands for *two-stage least squares*. It indicates that a model is
actually a sequence of two connected models.

1. **First Stage:** We regress the endogenous variable $X$ on the instrument $Z$ to
isolate the variation in $X$ that is "as good as random." This yields predicted
values $\hat{X}(Z)$.

2. **Second Stage:** We use these "clean" predictions to estimate $E(Y|\hat{X}(Z))$.
Since $\hat{X}(Z)$ is uncorrelated with the error term by construction, the
resulting coefficient provides a consistent causal estimate.

A naive implementation of 2SLS that uses two OLS models ignores that $\hat{X}$ is a
"generated regressor." OLS treats these predictions as fixed constants and fails to
account for the first-stage sampling uncertainty. This usually leads to
downward-biased (artificially small) standard errors.

We will overlook this nuance temporarily, but we will come back to it when we use
`IV2SLS`.

In [ ]:
# Declare first stage (stage 1; s1)
s1 = sm.OLS(
    endog=df['education'],
    exog=df[['qob_1', 'qob_2', 'qob_3'] + controls + fixed_effects]
).fit(cov_type='HC1')

# Use predictions from s1
df['education_inst'] = s1.fittedvalues

# Fit second stage
s2 = sm.OLS(
    endog=df['log_weekly_wage'],
    exog=df[['education_inst'] + controls + fixed_effects]
).fit(cov_type='HC1')

# Store results
education_manual = s2.params['education_inst']
print(f'Homemade estimated effect with IV2SLS: {education_manual:.3f}')
print(s2.summary())

## Official 2SLS

In [ ]:
# Declare and fit model
m2 = IV2SLS(
    dependent=df['log_weekly_wage'],
    exog=df[controls + fixed_effects],
    endog=df['education'],
    instruments=df[['qob_1', 'qob_2', 'qob_3']]
).fit()

# View results
print(m2.summary)